In [ ]:
!pip install reportlab
import gradio as gr
import uuid, os, tempfile, hashlib
from reportlab.lib.pagesizes import A5, landscape
from reportlab.pdfgen import canvas
from reportlab.lib.units import mm
from reportlab.lib.colors import HexColor
from datetime import datetime

# Certificate generation function (included for a self-contained script)
def generate_certificate(name, score, total, instructor="Baris Malcioglu"):
    unique_id = str(uuid.uuid4())
    filename = f"cert_{unique_id}.pdf"
    filepath = os.path.join(tempfile.gettempdir(), filename)
    c = canvas.Canvas(filepath, pagesize=landscape(A5))
    width, height = landscape(A5)
    c.setFillColor(HexColor("#fffdf6"))
    c.rect(0, 0, width, height, stroke=0, fill=1)
    c.setStrokeColor(HexColor("#001858"))
    c.setLineWidth(3)
    margin = 10 * mm
    c.rect(margin, margin, width - 2 * margin, height - 2 * margin)
    c.setFillColor(HexColor("#001858"))
    c.setFont("Helvetica-Bold", 24)
    c.drawCentredString(width / 2, height - 60, "Certificate of Completion")
    c.setFont("Helvetica", 14)
    c.drawCentredString(width / 2, height - 100, "This is awarded to")
    c.setFont("Helvetica-Bold", 18)
    c.drawCentredString(width / 2, height - 130, name)
    c.setFont("Helvetica", 14)
    c.drawCentredString(width / 2, height - 160, "For successfully completing the 1st quiz in KB608")
    c.setFont("Helvetica", 12)
    c.drawCentredString(width / 2, height - 185, f"Score: {score} / {total}")
    c.setFont("Helvetica-Oblique", 10)
    c.drawString(margin + 10, margin + 20, f"Instructor: {instructor}")
    date_str = datetime.now().strftime("%d %B %Y")
    c.setFont("Helvetica", 10)
    c.drawRightString(width - margin - 10, margin + 20, f"Issued on: {date_str}")
    c.save()
    return filepath

# Quiz data (answers are hashed)
quiz_type = "Text Answer"
questions = [{'question': 'How much memory in gigabytes does simulating 32 logical qubits require (at 8 bytes per float) ?', 'answer_hash': 'a68b412c4282555f15546cf6e1fc42893b7e07f271557ceb021821098dd66c1b'}]

def eval_quiz(name, *answers):
    """
    Evaluates the student's answers and generates a certificate only if the score is 80% or higher.
    """
    if not name.strip():
        name = "Anonymous"
    score = 0
    for i, ans in enumerate(answers):
        if ans and hashlib.sha256(str(ans).lower().strip().encode()).hexdigest() == questions[i]["answer_hash"]:
            score += 1

    total_questions = len(questions)
    passing_threshold = 0.8

    result_message = f"Hi {name}, your score is: {score} / {total_questions}."
    cert_path = None # Default to no certificate

    # Check if the score meets the passing threshold
    if total_questions > 0 and (score / total_questions) >= passing_threshold:
        cert_path = generate_certificate(name, score, total_questions, instructor="Baris Malcioglu")
        result_message += " Congratulations, you passed and earned a certificate!"
    else:
        result_message += " A score of 80% is required to receive a certificate."

    return result_message, cert_path

# Gradio interface for the student
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("## Test quiz")

    with gr.Row():
        name = gr.Textbox(label="Enter Your Full Name to Generate Certificate", placeholder="e.g., Ada Lovelace")

    answer_inputs = []
    for q in questions:
        gr.Markdown("**Question:** " + q['question'])
        if quiz_type == "Multiple Choice":
            answer_inputs.append(gr.Radio(choices=q["options"], label="Select your answer"))
        else:
            answer_inputs.append(gr.Textbox(label="Type your answer"))

    submit_btn = gr.Button("Submit Quiz")

    with gr.Row():
        result_output = gr.Textbox(label="Your Result")
        certificate_output = gr.File(label="Download Your Certificate")

    submit_btn.click(
        fn=eval_quiz,
        inputs=[name] + answer_inputs,
        outputs=[result_output, certificate_output]
    )

app.launch(debug=True)

Please enter your e-mails for gradescope integration:

[slido #1846638](https://app.sli.do/event/oYLPcXuYxxPe8baUNRu7Zf)

![Slido QR](https://github.com/osbama/Phys710/blob/main/theory/images/slidoPhys710.png?raw=1)


### The Image, Rank and Null space of a matrix



If you want to calculate how "large" will be the output space  of a linear map $A \in \mathcal{L}\left(\mathbb{C}^d\right)$, you first need to calculate the _image_ of $A$. The image of $A$ is the set of all possible output vectors under the action of $A$

$$ \text{Im}\left(A\right) := \left \{ \left|\psi\right\rangle\in  \mathbb{C}^d | \left|\psi\right\rangle=A\left|\phi\right\rangle \text{ for some }\left|\phi\right\rangle\in  \mathbb{C}^d    \right\} $$

The _rank_ of $A$ is the dimension of its image.

The set of all vectors sent to zero by $A$ is called its _null space_,

$$
\text{Null}\left(A\right) := \left \{ \left|\psi\right\rangle\in  \mathbb{C}^d | A\left|\phi\right\rangle=0 \right\}
$$


### Matrix Operations

Matrices encode the operations in general. There are some operations on matrices themselves as well. We will use frequently in this class the following such operators (linear maps):  _complex conjugate_ $\left(A^\star\right)$, _transpose_ $\left(A^T\right)$,  _adjoint_ $\left(A^\dagger\right)$ and _trace_ $\left(\text{Tr}(A)\right)$

The first three are defined as

$$
A^\star (i, j) = (A(i, j))^\star  \qquad A^T (i, j) = A(j, i) \qquad A^\dagger = (A^\star )^T
$$

Trace is simply a linear map $ \text{Tr :} \mathcal{L}(\mathbb{C}^d)\mapsto \mathbb{C} $  summing the entries on the diagonal of $A$.

$$
\text{Tr(A)} = \sum_{i=1}^d A(i, i).
$$

Trace is cyclic, i.e. $\text{Tr(ABC)} = \text{Tr}(CAB)$. This implies that even if A and B do not commute, i.e. $AB \neq BA$, it is still true that $\text{Tr}(AB) = \text{Tr}(BA)$.

Note that $(AB)^\dagger = B^\dagger A^\dagger$ , and $(AB)^T = B^T A^T$.

These operations apply to vectors as well

$$
\left\langle\psi\right|=\left|\psi\right\rangle^\dagger
$$


### Outer products.

You can use Dirac notation to describe matrices via _outer products_. For vectors $\left|\psi\right\rangle,\left|\phi\right\rangle \in \mathbb{C}^d $ , the outer product is $ \left|\psi\right\rangle\left\langle\phi\right| \in \mathcal L (\mathbb C ^d ) $ unlike the inner product, the outer product yields a $d \times d $ matrix. It can be computed straightforwardly via the rules of matrix multiplication. For example

$$
\left|0\right\rangle\left\langle 0 \right|= \begin{pmatrix} 1 \\ 0 \end{pmatrix} \begin{pmatrix} 1 & 0 \end{pmatrix} = \begin{pmatrix}
1 & 0\\
0 & 0
\end{pmatrix} \qquad
\left|1\right\rangle\left\langle 0 \right|= \begin{pmatrix} 0 \\ 1 \end{pmatrix} \begin{pmatrix} 1 & 0 \end{pmatrix} = \begin{pmatrix}
0 & 0\\
1 & 0
\end{pmatrix} \qquad
$$

More generally, the matrix  $ \left|i\right\rangle\left\langle j\right| \in \mathcal L (\mathbb C ^d ) $ has a 1 at position $(i, j)$ and zeroes elsewhere. This  yields a simple yet neat trick: A matrix $ A \in \mathcal L (\mathbb C ^d ) $  written in the computational basis can hence be expressed as $\sum_{ij} A(i, j)\left|i\right\rangle\left\langle j\right|$. It is thus easy to evaluate expressions of the form

$$
 \left\langle i \right| A\left|j\right\rangle =\left\langle i \right|\left(\sum_{i^\prime j^\prime}A(i^\prime,j^\prime)\left|i^\prime\right\rangle\left\langle j^\prime\right|\right)\left|j\right\rangle=\sum_{i^\prime j^\prime}A(i^\prime,j^\prime)\left\langle i \middle| i^\prime\right\rangle\left\langle j \middle| j^\prime\right\rangle=A(i,j)
$$

where the third equality follows since $\left\{\left| i\right\rangle\right\} $ forms an orthonormal basis for $\mathbb C^d$ . This method of obtaining matrix elements will be ubiquitous in the setting of quantum measurements.

### Eigenvalues and eigenvectors

With outer products in hand, we can discuss one of the most fundamental tools in our Linear Algebraic toolkit — eigenvalues and eigenvectors. Given any matrix $ A \in \mathcal L (\mathbb C ^d ) $ , an eigenvector is a special non-zero vector satisfying the equation

$$
A\left|\psi\right\rangle=\lambda\left|\psi\right\rangle
$$

for some $\lambda\in\mathbb C$ which is the corresponding _eigenvalue_.


The outer product can now be used to state an expression which will be used repeatedly in this course. For any matrix $A$ satisfying $AA^\dagger = A^\dagger A$ (such matrices are called normal ; most
matrices in this course will be normal), we can decompose $A$ in terms of its eigenvalues and eigenvectors as

$$
A=\sum_{i=1}^d \lambda_i \left|\lambda_i\right\rangle\left\langle\lambda_i\right|
$$

where $\lambda_i$ and $\left|\lambda_i\right\rangle$ are the eigenvalues and corresponding eigenvectors of $A$. This is called the _spectral decomposition_ of A. The spectral decomposition is useful for a few reasons. First, it
tells us exactly how $A$ acts on $\mathbb C^ d$ ; this is because the eigenvectors $\left|\lambda_i\right\rangle \in \mathbb C^d$ can be chosen to form an orthonormal basis for $\mathbb C^d$. Thus, any vector $|\left|psi\right\rangle \in \mathbb C^d$ can be written in terms of the eigenvectors of $A$, i.e. $\left|\psi\right\rangle=\sum_i \alpha_i \left|\lambda_i\right\rangle$   for some $\alpha_i \in \mathbb C$. The spectral decomposition also immediately reveals the rank of $A$; specifically, $\text{rank}(A)$ is just the number of non-zero eigenvalues of $A$. Finally, $\text{Tr}(A)$ has a very simple expression in terms of $A$’s eigenvalues,
$\text{Tr}(A) = \sum_i \lambda_i$ . Let us quickly prove this last claim:


$$
\text{Tr}(A) = \text{Tr}\left(\sum_{i=1}^d \lambda_i \left|\lambda_i\right\rangle\left\langle\lambda_i\right|\right)
=\sum_{i=1}^d \lambda_i \text{Tr}\left(\left|\lambda_i\right\rangle\left\langle\lambda_i\right|\right)
=\sum_{i=1}^d \lambda_i \text{Tr}\left(\left\langle\lambda_i\middle|\lambda_i\right\rangle\right)
=\sum_{i=1}^d \lambda_i
$$

Here, the second equality follows since the trace is linear, the third by the cyclic property of the trace, and the last since the eigenvectors $\left|\lambda_i\right\rangle$ are orthonormal.

Finally, recall that the eigenvalues of $A \in \mathcal L(\mathbb C^ d )$  can be computed as the roots of the degree-d _characteristic polynomial_ of $A$, $p_A$ , defined

$$
p_A(\lambda)=\text{det}\left(\lambda I - A\right)
$$

where the determinant det can be defined recursively as

$$
\text{det}(A) = \sum_{j=1}^d(−1)^{i+j} A(i, j) \text{det}(A_{ij})
$$

Here, $A_{ij}$ is the matrix obtained from $A$ by deleting row i and column j, and we define the base case of this recursion (i.e. a $1 \times 1$ matrix $[c]$) as $\text{det}([c]) = c$. This equation holds for any $i\in[d]$. In the special case when $d = 2$, this reduces nicely to

$$
\text{det}\begin{pmatrix} a & b \\ c & d \end{pmatrix}=ad-bc
$$

Let us close with a simple observation: For any diagonal matrix $A$ (written in the computational basis), the eigenvalues of $A$ are simply the entries on the diagonal of $A$, and the eigenvectors are just the computational basis vectors. In the exercise below, this immediately confirms that the eigenvalues of $Z$ are $1$ and $−1$ with eigenvectors $\left|0\right\rangle$ and $\left|1\right\rangle$, respectively.

## The four postulates of Quantum Mechanics

In this course, our aim is to study computing devices which operate according to the laws of quantum mechanics. Developed during the early 20th century by physicists Max Planck, Albert Einstein, Erwin Schrödinger and many others, quantum mechanics is a set of mathematical laws which describe the behaviour of subatomic particles such as protons, electrons, and photons. Although the theory has proven remarkably successful since its inception, it is nevertheless notoriously counterintuitive, an aspect which we shall explore in this lecture. Quantum mechanics is based on four postulates, which describe the following four intuitive ideas: How to describe a single quantum system, how to perform quantum operations on a quantum system, how to describe multiple quantum systems, and how to measure or extract classical information from a quantum system.

### Postulate 1: Individual Quantum systems

Recall that in the classical world, a bit $x$ can take on one of two values: 0 or 1. In the quantum world, we immediately see a radical departure from this statement, a quantum bit, or qubit, can take on not just 0 or 1, but rather both values 0 and 1 simultaneously. This is a very deep and counterintuitive statement, so it worth reflecting on:  it is like saying you can be both asleep and awake at the same time, or here on Earth and simultaneously on Mars at the same time. Indeed, relative to life as we know it, _it makes no sense!_

Let us formalize this phenomenon. We begin by encoding bits 0 and 1 via the standard basis vectors $\left|0\right\rangle, \left|1\right\rangle \in \mathbb C^2$ . Then, to denote that a qubit is in states $\left|0\right\rangle$ and $\left|1\right\rangle$ simultaneously, we write

$$
\left|0\right\rangle+\left|1\right\rangle.
$$

This is called a _superposition_. More generally, we can change the “extent” to which the qubit
is in state $\left|0\right\rangle$ versus $\left|0\right\rangle$ via _amplitudes_ $\alpha,\beta \in \mathbb C$ , i.e.

$$
\left|psi\right\rangle = \alpha\left|0\right\rangle + \beta\left|1\right\rangle
$$

The only restriction is that $\left|\psi\right\rangle$ must be a unit vector, i.e. that $\left|\alpha\right|^2+ \left|\beta\right|^2 = 1$. To summarize, any unit vector in $\mathbb C^2$ describes the state of a single qubit.

Qubit is a 2-dimensional (which describes a two state system). There is no physical limitation to use qubits: One can use d-state systems, resulting in d-dimensional states (called a qudit). Qudit is described by a unit vector $\left|\psi\right\rangle \in \mathbb C^d$ , which can be described as

$$
\left|\psi\right\rangle = \alpha_0 \left|0\right\rangle + \alpha_1 \left|1\right\rangle + \cdots + \alpha_{d-1} \left|d-1\right\rangle =\sum_{i=0}^{d-1}\alpha_i\left|i\right\rangle
$$

where $\left|i\right\rangle\in \mathbb C ^d$ denotes the _i_th computational basis vector and $\alpha_i \in \mathbb C$. Since $\left|\psi\right\rangle$ is a unit vector, we have $\sum_{i=0}^{d-1}\left|\alpha_i\right|^2=1$

### Postulate 2: Quantum operations

We next ask: What types of operations or maps can we perform on a qubit? Since a qubit is a vector, the natural choice is a linear map, i.e. multiplication by a matrix. However, not all matrices are fair game — it turns out that nature only allows a special class of matrices known as _unitary_ matrices. A unitary matrix $U \in \mathcal L( \mathbb C^ d )$ is one which satisfies $UU^\dagger = U^\dagger U = I$. In other words, the inverse of $U$ is simple to calculate — just take the dagger of $U$ . This immediately yields a key insight — all quantum gates (qubit operations) are _reversible_. Among the most common single qubit gates are the following, known as the _Pauli_ gates, after Austrian-Swiss physicist Wolfgang Pauli:

$$
X=
\begin{pmatrix}
0 & 1 \\
1 & 0
\end{pmatrix} \qquad
Y=
\begin{pmatrix}
0 & -i \\
i & 0
\end{pmatrix} \qquad
Z=
\begin{pmatrix}
1 & 0 \\
0 & -1
\end{pmatrix} \qquad
$$

#### $X$ Gate

The $X$ gate acts as a “quantum” NOT gate, as we see below:

$$
X\left|0\right\rangle=\begin{pmatrix}0 & 1 \\1 & 0\end{pmatrix}\begin{pmatrix}1  \\ 0 \end{pmatrix} =\begin{pmatrix}0  \\ 1 \end{pmatrix}=\left|1\right\rangle \qquad X\left|1\right\rangle=\begin{pmatrix}0 & 1 \\1 & 0\end{pmatrix}\begin{pmatrix}0  \\ 1 \end{pmatrix} =\begin{pmatrix}1  \\ 0 \end{pmatrix}=\left|0\right\rangle
$$

$\left|+\right\rangle$ and $\left|-\right\rangle$ are eigenvectors of $X$ i.e. $X\left|+\right\rangle=\left|+\right\rangle$ and $X\left|-\right\rangle=-\left|-\right\rangle$. The spectral decomposition of $X$ is hence

$$
X=\left|+\right\rangle\left\langle+\right|-\left|-\right\rangle\left\langle-\right|
$$

#### $Z$ Gate

The $Z$ gate, on the other hand, has no classical analogue. It acts as

$$
Z\left|0\right\rangle=\begin{pmatrix}1 & 0 \\0 & -1\end{pmatrix}\begin{pmatrix}1  \\ 0 \end{pmatrix} =\begin{pmatrix}1  \\ 0 \end{pmatrix}=\left|0\right\rangle \qquad Z\left|1\right\rangle=\begin{pmatrix}1 & 0 \\0 & -1\end{pmatrix}\begin{pmatrix}0  \\ 1 \end{pmatrix} =-\begin{pmatrix}0  \\ 1 \end{pmatrix}=-\left|1\right\rangle
$$

In other words, Z leaves $\left|0\right\rangle$ invariant, but injects a “phase” of −1 in front of $\left|1\right\rangle$. This also immediately shows that $\left|0\right\rangle$ and $\left|1\right\rangle$ are eigenvectors of Z with eigenvalues 1 and −1, respectively.

The Z gate is special in that it allows us to inject a relative phase into a quantum state. For example,

$$
Z\left|+\right\rangle=Z\left(\frac{1}{\sqrt 2} \left|0\right\rangle+\frac{1}{\sqrt 2}\left|1\right\rangle\right)=\frac{1}{\sqrt 2} Z \left|0\right\rangle+\frac{1}{\sqrt 2} Z \left|1\right\rangle=\frac{1}{\sqrt 2} \left|0\right\rangle-\frac{1}{\sqrt 2}\left|1\right\rangle=\left|-\right\rangle
$$

By relative phase, we mean that only the amplitude on $\left|1\right\rangle$ had its sign changed (or more generally, was multiplied by a phase $e^{i\pi} = −1$). If _all_ the amplitudes in the state were instead multiplied by $e^{i\pi}$ , then we could simply factor out the eiπ from the entire state — in this case,we would call $e^{i\pi}$ a global phase. It turns out that a global phase is insignificant in that it cannot be experimentally detected. A relative phase may seemingly also look unimportant - yet, as we shall see in this course, it is one of the features of quantum mechanics which allows quantum computers to outperform classical ones!

#### Hadamard Gate

Finally, we come to a fourth important unitary gate, the Hadamard gate:

$$
H=
\begin{pmatrix}
1 & 1 \\
1 & -1
\end{pmatrix}
$$

The Hadamard gate is special in that it creates superpositions for us. Namely, we have $H\left|0\right\rangle =\left|+\right\rangle$ and $H\left|1\right\rangle =\left|-\right\rangle$. It can also “erase” superpositions, i.e. $H\left|+\right\rangle =\left|0\right\rangle$ and $H\left|-\right\rangle =\left|1\right\rangle$ . In other words, $H$ is self-inverse — we have that $H^2 = I$ for $I$ the identity matrix. In fact, the Pauli matrices are also self-inverse.

#### Quantum Circuits

A _quantum circuit_ is a graphical model for depicting quantum computation. The quantum computation is represented by a sequence of quantum gates, measurements, initializations of qubits to known values, and etc. For example, here are three circuits:

![Three circuit examples](https://github.com/osbama/Phys710/blob/main/theory/images/Lecture3-fig1.png?raw=1)

They correspond to evolutions  $X\left|\psi\right\rangle $, $H\left|\psi\right\rangle$, and $HX\left|\psi\right\rangle $, respectively. Each wire in such a diagram denotes a quantum system, and a box labelled by gate $U$ depicts the action of unitary $U$ . We think of time going from left to right; for the last circuit above, note that the $X$ appears on the “left” in the circuit diagram but on the “right” in the expression $HX\left|\psi\right\rangle $; this is because $X$ should be applied first to $\left|\psi\right\rangle $, then $H$.

### Postulate 3: Composite Quantum Systems

Thus far, we have considered only single quantum systems, i.e. states $\left|\psi\right\rangle\in  \mathbb{C}^d$ for $d\geq 2$. We need multiple qubits interacting simultaneously for quantum computation. How can we mathematically describe, for example, the joint state of two qubits? The correct Linear Algebraic tool for this task is the tensor product, denoted $\otimes$. The tensor product allows us to “stitch together” two vectors. For example when two states $\left|\psi\right\rangle,\left|\phi\right\rangle\in  \mathbb{C}^2$ interact, it results in a larger 4-dimensional vector given by $\left|\psi\right\rangle\otimes\left|\phi\right\rangle\in  \mathbb{C}^4$ . Formally, we have $\mathbb{C}^2\otimes\mathbb{C}^2=\mathbb{C}^{2\times2}$ . In other words, the entries of a vector $\left|\psi\right\rangle\otimes\left|\phi\right\rangle\in  \mathbb{C}^2\otimes\mathbb{C}^2$  can be referenced via a pair of indices $(i, j)$ for $i, j \in {0, 1}$, and the specific rule for doing so is

$$
\left(\left|\psi\right\rangle\otimes\left|\phi\right\rangle\right)(i,j):=\psi_i\phi_j
$$

where recall $\psi_i$ and $\phi_j$ are the entries of $\left|\psi\right\rangle$ and $\left|\phi\right\rangle$, respectively. Here, you should think of the pair $(i, j)$ as representing the bits of a single index $x \in {0, 1, 2, 3}$. So for example, $(0, 0)$ is equivalent to index 0, (0, 1) to index 1, and (1, 1) to index 3. This implies that we can think of $\left|\psi\right\rangle\otimes\left|\phi\right\rangle$ as having four entries, i.e. $\left|\psi\right\rangle\otimes\left|\phi\right\rangle\in  \mathbb{C}^4$ . Let us demonstrate with some examples:

$$
\left|0\right\rangle\otimes\left|0\right\rangle= \begin{pmatrix}1  \\ 0 \end{pmatrix} \otimes \begin{pmatrix}1  \\ 0 \end{pmatrix}=\begin{pmatrix}1  \\ 0 \\ 0 \\ 0 \end{pmatrix} \qquad \left|0\right\rangle\otimes\left|1\right\rangle= \begin{pmatrix}1  \\ 0 \end{pmatrix} \otimes \begin{pmatrix}0  \\ 1 \end{pmatrix}=\begin{pmatrix}0  \\ 1 \\0\\0\end{pmatrix}
$$



Note that in the four equations above, the four-dimensional vectors obtained are just the computational basis vectors for $\mathbb{C}^4$ ! This hints at an important fact: If we take orthonormal bases $B_1 = \left\{\left|\psi_0\right\rangle, \left|\psi_1\right\rangle\right\}$ and  $B_2 = \left\{\left|\phi_0\right\rangle, \left|\phi_1\right\rangle\right\}$ , then we can obtain an orthonormal basis for $\mathbb{C}^4$ by tensoring together the elements of $B_1$ and $B_2$ in all four possible combinations, i.e. $\left\{\left|\psi_0\right\rangle\otimes\left|\phi_0\right\rangle, \left|\psi_0\right\rangle\otimes\left|\phi_1\right\rangle,\left|\psi_1\right\rangle\otimes\left|\phi_0\right\rangle,\left|\psi_1\right\rangle\otimes\left|\phi_1\right\rangle \right\}$ forms an orthonormal basis for $\mathbb{C}^4$ . For brevity, we shall often drop the notation $\otimes$ and simply write $\left|\psi\right\rangle\otimes\left|\phi\right\rangle=\left|\psi\right\rangle\left|\phi\right\rangle$.

Our discussion thus far generalizes straightforwardly to the case of $\mathbb{C}^{d_1}\otimes\mathbb{C}^{d_2}$ . Specifically, for $\left|\psi\right\rangle\in  \mathbb{C}^{d_1}$ and $\left|\phi\right\rangle\in  \mathbb{C}^{d_2}$ , we have that $\left|\psi\right\rangle\otimes\left|\phi\right\rangle\in\mathbb{C}^{d_1d_2}$. Then, for $i \in {0, . . . , d_1 − 1}$ and $j \in {0, . . . , d_2 − 1}$, we have $\left(\left|\psi\right\rangle\otimes\left|\phi\right\rangle\right)(i,j):=\psi_i\phi_j$ . Thus, for example, if we add a third qubit to our existing two qubit system, then we have a state which lives in $\mathbb{C}^{4}\otimes\mathbb{C}^{2}=\mathbb{C}^{8}$ . In fact, for each qubit we add to our system, the dimension grows by a factor of 2, i.e. it grows  exponentially — in general, an $n$-qubit state will correspond to a vector $\left|\psi\right\rangle\in  \mathbb{C}^{2^n}$! It is precisely this exponential growth in complexity which makes it difficult for classical computers to simulate the mechanics of an $n$-qubit quantum state — indeed, this was the reason why physicist Richard Feynman proposed the concept of a quantum computer in 1982 to begin with!

Finally, the tensor product has the following important properties for any $\left|a\right\rangle,\left|b\right\rangle\in  \mathbb{C}^{d_1}$ and $\left|c\right\rangle,\left|d\right\rangle\in  \mathbb{C}^{d_2}$ , which we will use repeatedly:

$$
\begin{align}
\left(\left|a\right\rangle+\left|b\right\rangle\right)\otimes\left|c\right\rangle &=\left|a\right\rangle\otimes\left|c\right\rangle+\left|b\right\rangle\otimes\left|c\right\rangle \\
\left|a\right\rangle\otimes \left(\left|c\right\rangle+\left|d\right\rangle\right) &=\left|a\right\rangle\otimes\left|c\right\rangle+\left|a\right\rangle\otimes\left|d\right\rangle \\
c \left(\left|a\right\rangle\otimes\left|c\right\rangle\right) &=\left(c\left|a\right\rangle\right)\otimes\left|c\right\rangle=\left|a\right\rangle\otimes\left(c\left|c\right\rangle\right) \\
\left(\left|a\right\rangle\otimes\left|c\right\rangle\right)^\dagger&=\left|a\right\rangle^\dagger\otimes\left|c\right\rangle^\dagger=\left\langle a\right|\otimes\left\langle c\right|\\
\left(\left\langle a\right|\otimes\left\langle c\right|\right)\left(\left|b\right\rangle\otimes\left|d\right\rangle\right)&=\left\langle a \middle| b\right\rangle\left\langle c \middle| d\right\rangle
\end{align}
$$

#### Quantum Entanglement

Now that we know how to stitch together a pair of single qubit states, it turns out we have opened Pandora’s box. For we can now talk about the two-qubit state which troubled Einstein to the end of his days — the innocuous-looking _Bell state_:

$$
\left|\Phi^+\right\rangle=\frac{1}{\sqrt 2}\left|0\right\rangle\left|0\right\rangle+\frac{1}{\sqrt 2}\left|1\right\rangle\left|1\right\rangle=\begin{pmatrix}\frac{1}{\sqrt 2}  \\ 0 \\0\\\frac{1}{\sqrt 2}\end{pmatrix}
$$
This state demonstrates a quantum phenomenon known as _entanglement_ — intuitively, if a pair $q_0$ and $q_1$ of qubits are entangled, then they are so “tightly bound” that one cannot accurately describe the state of $q_0$ or $q_1$ alone — only the _joint_ state of $q_0$ and $q_1$ can be described precisely. In the language of tensor products, this is captured by the following statement: There do not exist $\left|\psi_1\right\rangle,\left|\psi_2\right\rangle\in  \mathbb{C}^d$ such that $\left|\Phi^+\right\rangle=\left|\psi_1\right\rangle\otimes\left|\psi_2\right\rangle$. In 1935, Einstein, Podolsky and Rosen published a famous paper nowadays referred to as the “EPR” paper, in which they argue that quantum mechanics cannot be a complete physical theory because it allows the existence of states such as $\left|\Phi^+\right\rangle$. Fast forwarding a number of decades, we now not only believe entanglement is real, but we know that is is _necessary resource_ for quantum computers to outperform classical ones.

We shall later return to the topic of entanglement, but for now let us remark that there are three other such Bell states:

$$
\begin{align}
\left|\Phi^-\right\rangle&=\frac{1}{\sqrt 2}\left|00\right\rangle-\frac{1}{\sqrt 2}\left|11\right\rangle=\begin{pmatrix}\frac{1}{\sqrt 2}  \\ 0 \\0\\ -\frac{1}{\sqrt 2}\end{pmatrix}\\
\left|\Psi^+\right\rangle&=\frac{1}{\sqrt 2}\left|01\right\rangle+\frac{1}{\sqrt 2}\left|10\right\rangle=\begin{pmatrix}0  \\ \frac{1}{\sqrt 2} \\\frac{1}{\sqrt 2}\\0\end{pmatrix}\\
\left|\Psi^-\right\rangle&=\frac{1}{\sqrt 2}\left|01\right\rangle-\frac{1}{\sqrt 2}\left|10\right\rangle=\begin{pmatrix}0  \\ \frac{1}{\sqrt 2} \\-\frac{1}{\sqrt 2}\\0\end{pmatrix}\\
\end{align}
$$


Note that here we have further simplified notation by letting (e.g.) $\left|0\right\rangle\left|0\right\rangle=\left|00\right\rangle$. The four Bell states $\left\{\left|\Phi^+\right\rangle, \left|\Phi^-\right\rangle, \left|\Psi^+\right\rangle, \left|\Psi^-\right\rangle\right\}$ form an orthonormal basis for $\mathbb{C}^{4}$ known as the Bell basis, after Northern Irish physicist John Bell.


#### Two-qubit quantum gates.

We have seen that two-qubit quantum states are described by unit vectors in $\mathbb{C}^{4}$ . We can thus discuss two-qubit quantum gates, i.e. unitary operators $U \in\mathcal L (\mathbb{C}^{4})$. There are two types of such gates: The first are simply tensor products of one-qubit gates, such as $X \otimes Z$ or $H \otimes H$. Here, the tensor product is defined analogously for matrices as it is for vectors. (The formal description is cumbersome, but we follow with a helpful illustration to clarify.) For any $A \in\mathcal L (\mathbb{C}^{d_1})$,$B \in\mathcal L (\mathbb{C}^{d_2})$ , $A \otimes B$ is a $d_1d_2 \times d_1d_2$ complex matrix whose entries are indexed by $([d_1] \times [d_2], [d_1] \times [d_2])$ (where $[d] = \{0, . . . , d − 1\}$ here), such that

$$
(A\otimes B)((i_1,j_1),(i_2,j_2)):=A(i_1,i_2)B(j_1,j_2)
$$

To clarify this definition, suppose

$$
A=
\begin{pmatrix}
a_1 & a_2 \\
a_3 & a_4
\end{pmatrix}
\qquad
B=
\begin{pmatrix}
b_1 & b_2 \\
b_3 & b_4
\end{pmatrix}
$$

Then $A\otimes B$ is

$$
A \otimes B =
\begin{pmatrix}
a_1.\begin{pmatrix}
b_1 & b_2 \\
b_3 & b_4
\end{pmatrix} & a_2 . \begin{pmatrix}
b_1 & b_2 \\
b_3 & b_4
\end{pmatrix}\\
a_3. \begin{pmatrix}
b_1 & b_2 \\
b_3 & b_4
\end{pmatrix} & a_4. \begin{pmatrix}
b_1 & b_2 \\
b_3 & b_4
\end{pmatrix}
\end{pmatrix}
$$

In other words, $A\otimes B$ is obtained by taking four copies of $B$, each time multiplying by a different scalar entry of $A$.

The tensor product for matrices shares the properties of the tensor product for vectors, with the addition of two rules below:

$$
\begin{align*}
(A \otimes B)(C \otimes D) &= AC \otimes BD \\
Tr(A \otimes B) &= Tr(A)Tr(B).
\end{align*}
$$

In terms of circuit diagrams for tensor products of unitaries are like below: (We consider the cases of $X \otimes I$, $I \otimes Z$, and $H \otimes H$, respectively.)



Finally, we can also consider genuinely two-qubit gates, i.e. gates which are not the tensor product of single qubit gates. One important such gate is the controlled-NOT gate, denoted CNOT. The CNOT treats one qubit as the _control_ qubit, and the other as the target qubit. It then applies the Pauli $X$ gate to the target qubit only if the control qubit is set to $\left|1\right\rangle$. More precisely, the action of the CNOT on a two-qubit basis is given as follows, where qubit 1 is the control and qubit 2 is the target

$$
\text{CNOT}\left|00\right\rangle=\left|00\right\rangle \qquad \text{CNOT}\left|01\right\rangle=\left|01\right\rangle \qquad \text{CNOT}\left|10\right\rangle=\left|11\right\rangle \qquad \text{CNOT}\left|11\right\rangle=\left|10\right\rangle
$$

The CNOT gate is given by matrix:

$$
\text{CNOT}=
\begin{pmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 0 & 1 \\
0 & 0 & 1 & 0
\end{pmatrix}
=\begin{pmatrix}
 I & 0  \\
0 & X \\
\end{pmatrix}
$$

where the second expression is in block matrix form with $I$ and $X$ the identity and $X$ matrices.

The circuit diagram for the CNOT is given by

![CNOT](https://github.com/osbama/Phys710/blob/main/theory/images/Lecture3-fig4.png?raw=1)

With this in hand, we can do our first interesting computation — we can prepare the Bell state $\Phi^+$ starting from an initial state of $\left|00\right\rangle$ The preparation circuit is given as:

![Entangler](https://github.com/osbama/Phys710/blob/main/theory/images/Lecture3-fig5.png?raw=1)

## Delayed choice quantum eraser

A delayed-choice quantum eraser experiment, first performed by Yoon-Ho Kim, R. Yu, S. P. Kulik, Y. H. Shih and Marlan O. Scully, and reported in early 1999, is an elaboration on the quantum eraser experiment that incorporates concepts considered in Wheeler's delayed-choice experiment from late 70s to early 80s . The experiment was designed to investigate peculiar consequences of the well-known double-slit experiment in quantum mechanics, as well as the consequences of quantum entanglement.


![Delayed choice Quantum Eraser](https://github.com/osbama/Phys710/blob/main/theory/images/Delayed-Choice-Quantum-Eraser-1.jpg?raw=1)


 An argon laser generates individual 351.1 nm photons that pass through a double-slit apparatus (vertical black line in the upper left corner of the diagram).

An individual photon goes through one (or both) of the two slits. In the illustration, the photon paths are color-coded as red or green lines to indicate which slit the photon came through (red indicates slit A, green indicates slit B).

So far, the experiment is like a conventional two-slit experiment. However, after the slits, spontaneous parametric down-conversion (SPDC) is used to prepare an entangled two-photon state. This is done by a nonlinear optical crystal BBO (beta barium borate) that converts the photon (from either slit) into two identical, orthogonally polarized entangled photons with $\frac{1}{2}$ the frequency of the original photon. The paths followed by these orthogonally polarized photons are caused to diverge by the Glan–Thompson prism.

One of these 702.2 nm photons, referred to as the "signal" photon (look at the red and green lines going upwards from the Glan–Thompson prism) continues to the target detector called $D_0$. During an experiment, detector $D_0$ is scanned along its x axis, its motions controlled by a step motor. A plot of "signal" photon counts detected by $D_0$ versus x can be examined to discover whether the cumulative signal forms an interference pattern.

The other entangled photon, referred to as the "idler" photon (look at the red and green lines going downwards from the Glan–Thompson prism), is deflected by prism PS that sends it along divergent paths depending on whether it came from slit A or slit B.

Somewhat beyond the path split, the idler photons encounter beam splitters $S_a$, $S_b$, and $S_c$ that each have a 50% chance of allowing the idler photon to pass through and a 50% chance of causing it to be reflected. $M_a$ and $M_b$ are mirrors.

The beam splitters and mirrors direct the idler photons towards detectors labeled $D_1$, $D_2$, $D_3$ and $D_4$. Note that:

* If an idler photon is recorded at detector $D_3$, it can only have come from slit B.
* If an idler photon is recorded at detector $D_4$, it can only have come from slit A.
* If an idler photon is detected at detector $D_1$ or $D_2$, it might have come from slit A or slit B.
* The optical path length measured from slit to $D_1$, $D_2$, $D_3$ and $D_4$ is 2.5 m longer than the optical path length from slit to $D_0$. This means that any information that one can learn from an idler photon must be approximately 8 ns later than what one can learn from its entangled signal photon.

Let's call the green path $\left|0\right\rangle$ and red path $\left|1\right\rangle$. The double slit can be described by a Hadamard gate. The SPDC corresponds to a CNOT-gate. The state after the SPDC $\frac{1}{2} \left(\left|00\right\rangle+\left|11\right\rangle\right)$. There is a phase $\phi$ depending on the position $x$ of the detector $D_0$, which corresponds to a phase gate $R_\phi$. The state before the measurement is

$$
 \frac{1}{2} \left(\left|00\right\rangle+\left|10\right\rangle+e^{i\phi}\left|01\right\rangle-e^{i\phi}\left|11\right\rangle\right)
 $$
 $$
 =\frac{1}{2} \left(  \left(\left(1+e^{i\phi}\right)\left|0\right\rangle+\left(1-e^{i\phi}\right)\left|1\right\rangle \right)\left|+\right\rangle+\left(\left(1-e^{i\phi}\right)\left|0\right\rangle+\left(1+e^{i\phi}\right)\left|1\right\rangle \right)\left|-\right\rangle   \right)
 $$

Detection of the idler photon by $D_3$ or $D_4$ provides delayed "which-path information" indicating whether the signal photon with which it is entangled had gone through slit A or B. On the other hand, detection of the idler photon by $D_1$ or $D_2$ provides a delayed indication that such information is not available for its entangled signal photon. Insofar as which-path information had earlier potentially been available from the idler photon, it is said that the information has been subjected to a "delayed erasure".

* When the experimenters looked at the signal photons whose entangled idlers were detected at $D_1$ or $D_2$, they detected interference patterns.
* However, when they looked at the signal photons whose entangled idlers were detected at $D_3$ or $D_4$, they detected simple diffraction patterns with no interference.


Let's look at the probability to measure the a photon in $D_0$.

* If we measure the second photon in $D_3$ or $D_4$, the post measurement state collapses to $\left|\pm\right\rangle$ basis. Then the probability for a click in $D_0$ is $\frac{1}{2}$ . This is independent of the phase: no interference.
* If we measure the second photon in $D_1$ or $D_2$, the probability for a click at $D_0$ is $\frac{1}{2}\left(1\mp \cos\phi\right)$, so here we see the interference. Whether we see interference or not depends on the basis choice on the second system, which can be delayed.

![Oversimplified delayed choice quantum eraser](https://github.com/osbama/Phys710/blob/main/theory/images/oversimplified_delayed_choice_eraser_quantum.png?raw=1)

![Simulated recordings](https://github.com/osbama/Phys710/blob/main/theory/images/KimDelayedChoiceQuantumEraserGraphsGIF.gif?raw=1)
Simulated joint detection rates between $D_0$ and $D_1$, $D_2$, $D_3$, $D_4$ ($R_{01}$, $R_{02}$, $R_{03}$, $R_{04}$).



![Quantum Eraser with coincidence counter](https://github.com/osbama/Phys710/blob/main/theory/images/Kim_EtAl_Quantum_Eraser.svg.png?raw=1)


As you can see I have omitted the "coincidence counter" from the above description, similar to many people who talked about this experiment. Actually  this is a key element. The coincidence counter would appear as a purely electronic device that acts on measurements. So if you would cast it as a quantum circuit it would look like this:

![Coincidence counter Reveal case ](https://github.com/osbama/Phys710/blob/main/theory/images/cycle-revealed.gif?raw=1)
![Coincidence counter Erasure case](https://github.com/osbama/Phys710/blob/main/theory/images/cycle-erased.gif?raw=1)

The data was always the same! The "erasure" only changes how we *analyse* the data, i.e. what we keep and what we discard. This is done after the measurement, using an electronic circuit. There's no time travel, no disproof of materialism, just plain 'ol correlation being mistaken for causation.

A full circuit can be found [here for reference purposes](https://algassert.com/quirk#circuit=%7B%22cols%22%3A%5B%5B1%2C%22H%22%5D%2C%5B1%2C%22%E2%80%A2%22%2C1%2C1%2C%22X%22%5D%2C%5B1%2C%22%7Eslits%22%2C%22QFT7%22%5D%2C%5B1%2C1%2C%22Measure%22%2C%22Measure%22%2C%22Measure%22%2C%22Measure%22%2C%22Measure%22%2C%22Measure%22%2C%22Measure%22%5D%2C%5B%22%E2%80%A6%22%2C%22%E2%80%A6%22%2C%22Chance7%22%5D%2C%5B%22%E2%80%A6%22%2C%22%E2%80%A6%22%5D%2C%5B%22%E2%80%A6%22%2C%22%E2%80%A6%22%5D%2C%5B%22%E2%80%A6%22%2C%22%E2%80%A6%22%5D%2C%5B%22H%22%5D%2C%5B%22Measure%22%5D%2C%5B%22%7Echoice%22%5D%2C%5B%22%E2%80%A2%22%2C%22X%5E%C2%BD%22%5D%2C%5B1%2C%22Measure%22%5D%2C%5B1%2C%22%7Eresult%22%2C1%2C1%2C1%2C%22%7Eflat%22%5D%2C%5B%22%E2%97%A6%22%2C%22%E2%97%A6%22%2C%22Chance7%22%5D%2C%5B%22%E2%97%A6%22%2C%22%E2%80%A2%22%2C%22Chance7%22%5D%2C%5B1%2C1%2C1%2C1%2C1%2C%22%7Ewaves%22%5D%2C%5B%22%E2%80%A2%22%2C%22%E2%97%A6%22%2C%22Chance7%22%5D%2C%5B%22%E2%80%A2%22%2C%22%E2%80%A2%22%2C%22Chance7%22%5D%5D%2C%22gates%22%3A%5B%7B%22id%22%3A%22%7Echoice%22%2C%22name%22%3A%22choice%22%2C%22matrix%22%3A%22%7B%7B1%2C0%7D%2C%7B0%2C1%7D%7D%22%7D%2C%7B%22id%22%3A%22%7Eresult%22%2C%22name%22%3A%22result%22%2C%22matrix%22%3A%22%7B%7B1%2C0%7D%2C%7B0%2C1%7D%7D%22%7D%2C%7B%22id%22%3A%22%7Eflat%22%2C%22name%22%3A%22flat%22%2C%22matrix%22%3A%22%7B%7B1%2C0%7D%2C%7B0%2C1%7D%7D%22%7D%2C%7B%22id%22%3A%22%7Ewaves%22%2C%22name%22%3A%22waves%22%2C%22matrix%22%3A%22%7B%7B1%2C0%7D%2C%7B0%2C1%7D%7D%22%7D%2C%7B%22id%22%3A%22%7Eslits%22%2C%22name%22%3A%22slits%22%2C%22matrix%22%3A%22%7B%7B1%2C0%7D%2C%7B0%2C1%7D%7D%22%7D%5D%7D)

In fact you can even make classical analogues:
Bob has a six-sided die, a two-sided coin, and a small empty box to put the coin in. He rolls the die, gets a result between 1 and 6, and places the coin in the box in a way that depends on the die roll. If the die roll was even, the coin is placed heads up. If the die roll was odd, the coin is placed tails up. Bob then writes down his die roll, and carefully hands the box to Alice.

Alice now has to decide to either a) just open the box or b) shake the box before opening it. (Shaking the box randomizes the coin.) Once the box is open, she writes down whether the coin was face up or face down, and also writes down whether or not she shook the box.

![Classical erasure](https://github.com/osbama/Phys710/blob/main/theory/images/classical-banded-choice.png?raw=1)


The tweaked delayed choice quantum eraser so that there isn't a classical analogue is called a [Bell test](https://en.wikipedia.org/wiki/Bell_test), however it lacks the "characteristics" of the delayed choice quantum eraser.  For example, in order to avoid the signalling loophole, the choice can no longer be delayed.

## Postulate 4: Measurement

How do we mathematically model the act of measuring or observing a quantum system considering that the very act of looking at or observing a quantum system irreversibly alters the state of the system? To model this phenomenon, we shall use the notion of a projective or von Neumann measurement (named after Hungarian child prodigy and mathematician, John von Neumann, who was apparently already familiar with calculus at the age of 8). To do so, we must define three classes of linear operators, each of which is increasingly more restricted. All three classes will prove vital throughout this course.

 1. *Hermitian operators*: An operator $M \in \mathcal L \left(\mathbb C^d \right)$ is Hermitian if $M = M^\dagger$ . Examples you are already familiar with are the Pauli $X$, $Y$ , and $Z$ gates, which are not only unitary, but also Hermitian. A Hermitian operator has the important property that all of its eigenvalues are real. Thus, Hermitian operators can be thought of as a higher dimensional generalization of the real numbers.
 2. *Positive semi-definite operators*: If a Hermitian operator has only non-negative eigenvalues, then it is called positive-semidefinite. Thus, positive semi-definite (or positive for short) matrices generalize the non-negative real numbers.
 3. *Orthogonal projection operators*: A Hermitian matrix $\Pi \in \mathcal L \left(\mathbb C^d \right)$ is an orthogonal projection operator (or projector for short) if $\Pi^2=\Pi$. This is equivalent to saying $\Pi$ has only eigenvalues 0 and 1. Let us prove this equivalence briefly: Since $\Pi$ is Hermitian, we can take its spectral decomposition, $\Pi = \sum_i \lambda_i \left|\lambda_i\right\rangle\left\langle\lambda_i\right|$. Hence,
   $$
    \sum_i \lambda_i \left|\lambda_i\right\rangle\left\langle\lambda_i\right|=\Pi =\Pi^2=\left(\sum_i \lambda_i \left|\lambda_i\right\rangle\left\langle\lambda_i\right|\right)\left(\sum_i \lambda_i \left|\lambda_i\right\rangle\left\langle\lambda_i\right|\right)=\sum_i \lambda_i^2 \left|\lambda_i\right\rangle\left\langle\lambda_i\right|
   $$
   where the last equality follows since {|λi i} is an orthonormal basis. Since the |λi i are orthogonal, we thus have that for all i, λi = λ2i . But this can only hold if λi ∈ {0, 1}, as claimed.

   It is important to note that since a projector $\Pi$’s eigenvalues are all 0 or 1, its spectral decomposition must take the form $\Pi=\sum_i \left|\psi_i\right\rangle\left\langle\psi_i\right|$, where $\left\{\left|\psi_i\right\rangle\right\}$ are an orthonormal set. Conversely, summing any set of orthonormal $\left\{\left|\psi_i\right\rangle\right\}$ in this fashion yields projector. Observe that a projector $\Pi$ has rank 1 if and only if $\Pi=\left|\psi\right\rangle\left\langle\psi\right|$ for some $\left|\psi_i\right\rangle \in \mathbb C ^ d $ , since the rank of $\Pi$ equals the number of non-zero eigenvalues of $\Pi$, and here $\Pi=\left|\psi\right\rangle\left\langle\psi\right|$ is a spectral decomposition of $\Pi$. Finally,let us develop an intuition for what a projector actually *does* — for any projector $\Pi=\sum_i \left|\psi_i\right\rangle\left\langle\psi_i\right|$ and state $\left|\phi\right\rangle$ to be measured, we have
   $$
   \Pi\left|\phi\right\rangle=\left(\sum_i \left|\psi_i\right\rangle\left\langle\psi_i\right|\right)\left|\phi\right\rangle=\sum_i \left|\psi_i\right\rangle\left(\left\langle\psi_i\middle|\phi\right\rangle\right)=\sum_i \left(\left\langle\psi_i\middle|\phi\right\rangle\right)\left|\psi_i\right\rangle\in\text{Span}\left\{\left|\psi_i\right\rangle\right\}
   $$

   where note $\left\langle\psi_i\middle|\phi\right\rangle\in \mathbb C$. Thus, $\Pi$ projects us down onto the span of the vectors $\left\{\left|\psi_i\right\rangle\right\}$.

### Projective Measurements.

With projectors in hand, we can now define a projective measurement. A *projective measurement* is a set of projectors $B = \left\{\Pi_i\right\}_{i=0}^m$ such that $\sum_{i=0}^m \Pi_i = I$. The latter condition is called the completeness relation. If each $\Pi_i$ is rank one, i.e. $\Pi_i=\left|\psi_i\right\rangle\left\langle\psi_i\right|$, then we say that $B$ models a *measurement in basis* $\left\{\left|\psi_i\right\rangle\right\}$. Often, we shall measure in the computational basis, which is specified by $B= \left\{\left|0\right\rangle\left\langle0\right|,\left|1\right\rangle\left\langle1\right|\right\}$ in the case of $\mathbb C ^ 2$  (and generalizes as $B= \left\{\left|i\right\rangle\left\langle,\right|\right\}_{i=0}^{d-1}$ for $\mathbb C^d$)

With a projective measurement $B = \left\{\Pi_i\right\}_{i=0}^m\subseteq\mathbb C^d$ in hand, let us *specify* how one uses $B$. Suppose our quantum system is in state $\left|\psi\right\rangle\in\mathbb C^d$ . Then, the probability of obtaining outcome $i \in \{0, . . . , m\}$ when measuring $\left|\psi\right\rangle$ with $B$ is given by
$$
\text{Pr}(\text{outcome } i) = \text{Tr}\left(\Pi_i \left|\psi\right\rangle \left\langle\psi\right|\Pi_i \right) = \text{Tr}\left(\Pi_i^2 \left|\psi\right\rangle \left\langle\psi\right|\right) = \text{Tr}\left(\Pi_i \left|\psi\right\rangle \left\langle\psi\right|\right),
$$
where the second equality follows by the cyclic property of the trace and the third since $\Pi_i$ is
a projector.

The exercise above has an important moral — requiring a quantum state $\left|\psi\right\rangle$ to be a unit vector (i.e. $\left|\alpha\right|^2 + \left|\beta\right|^2 = 1$) ensures that when measuring $\left|\psi\right\rangle$, the distribution over the outcomes is a valid probability distribution, i.e. the probabilities for all possible outcomes sum to 1. The other important take-home message here is that measurements in quantum mechanics are inherently probabilistic — in general, the outcomes cannot be perfectly predicted! Finally, we started this lecture by saying that the very act of measuring a quantum state disturbs the system. Let us now formalize this; we will crucially use the fact discussed earlier that a projector projects a vector $\left|\psi\right\rangle$ down into a smaller subspace. Specifically, upon obtaining outcome $\Pi_i$ when measuring $B$, the state of system “collapses” to

$$
\frac{\Pi_i \left|\psi\right\rangle \left\langle\psi\right|\Pi_i }{\text{Tr}\left(\Pi_i \left|\psi\right\rangle \left\langle\psi\right|\Pi_i \right)}=\frac{\Pi_i \left|\psi\right\rangle \left\langle\psi\right|\Pi_i }{\text{Tr}\left(\Pi_i \left|\psi\right\rangle \left\langle\psi\right| \right)}
$$

Note the denominator above is a scalar, and is just the probability of outcome $i$. There are two points here which may confuse you: Why have we written the output state as a matrix $\Pi_i \left|\psi\right\rangle \left\langle\psi\right|\Pi_i$ rather than a vector $\Pi_i \left|\psi\right\rangle$, and what is the role of the denominator? Let us handle each of these in order.

First, conditioned on outcome $\Pi_i$ , the output state is indeed a vector, namely $\Pi_i \left|\psi\right\rangle$. However, there is a more general formalism which we shall discuss shortly called the *density operator formalism*, in which quantum states are written as matrices, not vectors. Specifically, the “density matrix” representing vector $\left|\psi\right\rangle$ would be written as matrix $\left|\psi\right\rangle \left\langle\psi\right|$. The density operator formalism is more general than the state vector approach we have taken so far, and will be crucial for studying individual subsystems of a larger composite quantum state. Thus, the answer to question 1 is that we have written the output as a matrix simply to slowly ease the transition into the density matrix formalism.

The motive behind question 2 is somewhat less sinister — the problem here is that since we projected out part of $\left|\psi\right\rangle$ during the measurement, the output $\Pi_i \left|\psi\right\rangle$  may not necessarily be normalized. To renormalize $\Pi_i \left|\psi\right\rangle$, we simply divide by its Euclidean norm to obtain

$$
\left|\psi^\prime\right\rangle=\frac{\Pi_i \left|\psi\right\rangle}{\left\|\Pi_i \left|\psi\right\rangle\right\|_2}=\frac{\Pi_i \left|\psi\right\rangle}{\sqrt{ \left\langle\psi\right| \Pi_i \Pi_i \left|\psi\right\rangle}}=\frac{\Pi_i \left|\psi\right\rangle}{\sqrt{ \left\langle\psi\right| \Pi_i \left|\psi\right\rangle}}
$$

The state $\left|\psi^\prime\right\rangle$ describes the post-measurement state of our system, assuming we have obtained outcome $i$.

A final quirk we should iron out is the following — in terms of measurements, what is the consequence of the fact that a projector $\Pi_i$ satisfies $\Pi_i^2 = \Pi_i$ ? Well, if you observe a quantum system now, and then again five minutes from now, and if the system has not been subjected to any gates or noise in between the measurements, then the two measurement results you obtain should agree (I realize the study of quantum mechanics has likely distorted your view of when you can trust your intuition, but this is one case in which you can). To model this, suppose we measure using $B = \{\Pi_i\}$ and obtain results $i$ and $j$ in measurements 1 and 2, respectively. Then:

$$
\text{Pr}\left(\text{outcome } i \middle| \text{outcome } j\right)=\text{Tr}\left(\Pi_j\frac{\Pi_i \left|\psi\right\rangle \left\langle\psi\right|\Pi_i }{\text{Tr}\left(\Pi_i \left|\psi\right\rangle \left\langle\psi\right| \right)}\Pi_j\right)=\frac{\text{Tr}\left(\Pi_j\Pi_i \left|\psi\right\rangle \left\langle\psi\right|\Pi_i\Pi_j \right)}{\text{Tr}\left(\Pi_i \left|\psi\right\rangle \left\langle\psi\right| \right)}
$$

To simplify this expression, we use the fact that if the completeness relation holds for projectors $\{\Pi_i\}$, i.e. $\sum_i \Pi_i = I$, then it turns out that $\Pi_j \Pi_1  = \delta_{ij} \Pi_i$, where recall $\delta_ij$ is the Kronecker delta. Thus, if $i\neq j$, above equation equals to 0, and if $i = j$, it reduces to
$$
\frac{\text{Tr}\Pi_j\Pi_i \left|\psi\right\rangle \left\langle\psi\right|\Pi_i\Pi_j }{\text{Tr}\left(\Pi_i \left|\psi\right\rangle \left\langle\psi\right| \right)}=\frac{\text{Tr}\left(\Pi_i^2 \left|\psi\right\rangle \left\langle\psi\right|\Pi_i^2 \right)}{\text{Tr}\left(\Pi_i \left|\psi\right\rangle \left\langle\psi\right| \right)}=\frac{\text{Tr}\left(\Pi_i \left|\psi\right\rangle \left\langle\psi\right|\Pi_i\right) }{\text{Tr}\left(\Pi_i \left|\psi\right\rangle \left\langle\psi\right| \right)}=\frac{\text{Tr}\left(\Pi_i \left|\psi\right\rangle \left\langle\psi\right|\right)}{\text{Tr}\left(\Pi_i \left|\psi\right\rangle \left\langle\psi\right| \right)}=1
$$
i.e. measuring the state a second time again yields outcome i with probability 1, as desired.
Thus, although observing a state for the first time disturbs it, subsequent measurements will
consistently return the same measurement result!

We close this section by giving the circuit symbol which denotes a measurement of a qubit $\left|\psi\right\rangle \in \mathbb C^2$ in the computational basis $B= \left\{\left|0\right\rangle\left\langle0\right|,\left|1\right\rangle\left\langle1\right|\right\}$:

![Measurement circuit](https://github.com/osbama/Phys710/blob/main/theory/images/measure_circuit.png?raw=1)

The double-wires on the right side indicate that the output of the measurement is a classical string (indicating which measurement outcome was obtained).

## Quantum Teleportation

With the concepts of the Bell state and measurement in hand, we can discuss our first neat computational trick: Quantum teleportation. Suppose you have a single-qubit quantum state $\left|\psi\right\rangle= \alpha\left|0\right\rangle + \beta\left|1\right\rangle $ in your possession (i.e. as a physical system, not on paper), but that you do not know the values of $\alpha$ and $\beta$. Your friend Alice now phones you and asks to borrow your state. How can you send it to her? One obvious way is simply to pop your system in the mail and physically send it over. However, it turns out that by exploiting the phenomenon of entanglement, you can do something incredible — by sending two classical bits over the telephone to Alice, you can “teleport” $\left|\psi\right\rangle$ instantly to her!

To teleport $\left|\psi\right\rangle$, we assume that you and Alice each share half of a Bell state $\left|\Phi^+\right\rangle=\frac{1}{\sqrt 2}\left(\left|00\right\rangle+\left|11\right\rangle\right)$ to begin with; specifically, you hold qubit 1 of $\left|\Phi^+\right\rangle$, and Alice holds qubit 2. The teleportation circuit is then given as follows:

![The teleportation circuit](https://github.com/osbama/Phys710/blob/main/theory/images/teleportation_circuit.png?raw=1)

Let us break this down piece by piece. The first two wires are held by you; wire 1 holds the state to be teleported, $\left|\psi\right\rangle$, and wire 2 holds your half of $\left|\Phi^+\right\rangle$. The third wire holds Alice’s half of $\left|\Phi^+\right\rangle$. Note that we have used $\left|\Phi^+_A\right\rangle$ and $\left|\Phi^+_B\right\rangle$ to denote the two “halves” of $\left|\Phi^+\right\rangle$, but this is poor notation — read literally, this diagram suggests $\left|\Phi^+\right\rangle=\left|\Phi^+_A\right\rangle\otimes\left|\Phi^+_B\right\rangle$, which is not true since is $\left|\Phi^+\right\rangle$ entangled, and hence from last lecture we know that there do not exist states $\left|\Phi^+_A\right\rangle$ and $\left|\Phi^+_B\right\rangle$ such that $\left|\Phi^+\right\rangle=\left|\Phi^+_A\right\rangle\otimes\left|\Phi^+_B\right\rangle$ This notation is for illustration purposes only, not to complicate the diagram, and make our analysis easy.

The circuit can be divided into 5 steps: Step 1 performs the CNOT, Step 2 the Hadamard gate, Step 3 measures qubits 1 and 2, Step 4 applies a conditional X gate, and Step 5 applies a conditional Z gate. The latter two require clarification: The conditional X gate here takes a classical bit $b$ as input (hence the incoming wire at the top is a double line), and applies X if and only if $b = 1$. The conditional Z gate is defined analogously.

Now that we have technically parsed this diagram, let us intuitively parse it. First, you begin in Steps 1 and 2 by performing a CNOT and Hadamard on your qubits, followed by a standard basis measurement in Step 3. Since a measurement in the standard basis maps each qubit to either $\left|0\right\rangle$ or $\left|1\right\rangle$, the output of your two measurements can jointly be thought of as one of the four bit strings $00$, $01$, $10$, or $11$. Call these bits $b_0 b_1$ . Now you pick up the telephone, call Alice, and tell her the value of $b_0 b_1$ . Conditioned on $b_0 $ , she applies X to her half of the Bell pair, followed by Z conditioned on $b_1$. The claim is that at this point, Alice’s qubit’s state has been magically converted to $\left|\psi\right\rangle$. In fact, as we shall see shortly, $\left|\psi\right\rangle$ has also disappeared from your possession! In this sense, teleportation has taken place.

Let us formally analyze the action of this circuit. Denote by $\left|\psi_1\right\rangle$ for $i \in \{0, . . . , 5\}$ the joint state of your and Alice’s systems immediately after Step $i$ has taken place. Here, we define $\left|\psi_0\right\rangle$ as the initial joint state before any gates are applied; it is given by

$$
\begin{align}
\left|\psi_0\right\rangle=\left|\psi\right\rangle\left|\Phi^+\right\rangle=\left(\alpha\left|0\right\rangle + \beta\left|1\right\rangle\right)\left(\frac{1}{\sqrt 2}\left|00\right\rangle+\frac{1}{\sqrt 2}\left|11\right\rangle\right)=\frac{1}{\sqrt 2}\left(\alpha\left|000\right\rangle+\alpha\left|011\right\rangle+\beta\left|100\right\rangle+\beta\left|111\right\rangle\right)
\end{align}
$$

After Step 1, i.e. after the CNOT, we have state

$$
\frac{1}{\sqrt 2}\left(\alpha\left|000\right\rangle+\alpha\left|011\right\rangle+\beta\left|110\right\rangle+\beta\left|101\right\rangle\right)
$$

After Step 2, i.e. after the Hadamard gate, we have

$$
\begin{align}
&\frac{1}{\sqrt 2}\left(\alpha\left|+\right\rangle\left|00\right\rangle+\alpha\left|+\right\rangle\left|11\right\rangle+\beta\left|-\right\rangle\left|10\right\rangle+\beta\left|-\right\rangle\left|01\right\rangle\right)\\
&=\frac{1}{2}\left(\alpha(\left|0\right\rangle+\left|1\right\rangle)\left|00\right\rangle+\alpha(\left|0\right\rangle+\left|1\right\rangle)\left|11\right\rangle+\beta(\left|0\right\rangle-\left|1\right\rangle)\left|10\right\rangle+\beta(\left|0\right\rangle-\left|1\right\rangle)\left|01\right\rangle\right)\\
&=\frac{1}{2}\left(\left|00\right\rangle\left(\alpha\left|0\right\rangle+\beta\left|1\right\rangle\right)+\left|01\right\rangle\left(\alpha\left|0\right\rangle+\beta\left|1\right\rangle\right)+\left|10\right\rangle\left(\alpha\left|0\right\rangle-\beta\left|1\right\rangle\right)+\left|11\right\rangle\left(\alpha(\left|1\right\rangle-\beta\left|0\right\rangle\right)\right)
\end{align}
$$

Let us now pause and analyze the state of affairs. There are four terms in this superposition, each of which begins with a distinct bit string $\left|00\right\rangle$, $\left|01\right\rangle$, $\left|10\right\rangle$, or $\left|11\right\rangle$. This means that if you now measure qubits 1 and 2 in the standard basis and obtain outcome (say) $\left|00\right\rangle$, then Alice’s qubit on wire 3 collapses to the only consistent possibility, $\alpha\left|0\right\rangle + \beta\left|1\right\rangle$. In this case, teleportation has already taken place.

More generally, the four possible outcomes upon measuring qubits 1 and 2 result in four distinct residual states on Alice’s qubit as follows:
$$
00\rightarrow\alpha\left|0\right\rangle + \beta\left|1\right\rangle\qquad 01\rightarrow\alpha\left|1\right\rangle + \beta\left|0\right\rangle\qquad 10\rightarrow\alpha\left|0\right\rangle - \beta\left|1\right\rangle\qquad 11\rightarrow\alpha\left|1\right\rangle - \beta\left|0\right\rangle
$$

Thus, if you simply send the two bits $b_0 b_1$ encoding the measurement outcome to Alice, then regardless of the value of $b_0 b_1$ , she can recover your original state $\left|\psi\right\rangle=\alpha\left|0\right\rangle + \beta\left|1\right\rangle $  via the following identities:

$$
X\left(\alpha\left|1\right\rangle + \beta\left|0\right\rangle\right)=\alpha\left|0\right\rangle + \beta\left|1\right\rangle \qquad Z\left(\alpha\left|0\right\rangle - \beta\left|1\right\rangle\right)=\alpha\left|0\right\rangle + \beta\left|1\right\rangle \qquad
ZX\left(\alpha\left|1\right\rangle - \beta\left|0\right\rangle\right)=\alpha\left|0\right\rangle + \beta\left|1\right\rangle
$$

In other words, by conditionally applying X and Z based on the outputs $b_0 b_1$ from your measurement, Alice can successfully recover your state $\left|\psi\right\rangle=\alpha\left|0\right\rangle + \beta\left|1\right\rangle $ . This is precisely what is depicted in Steps 4 and 5 of the teleportation circuit. Finally, note that since measuring your qubits leaves you in one of the four standard basis states $\left|00\right\rangle$, $\left|01\right\rangle$, $\left|10\right\rangle$, or $\left|11\right\rangle$ the state $\left|\psi\right\rangle$ has now “disappeared” from your possession!